# 🛡️ Sentinela — Servidor MedGemma

Este notebook inicia o servidor MedGemma e expõe um endpoint via ngrok.

## Pré-requisitos
1. Aceite os termos do MedGemma em: https://huggingface.co/google/medgemma-4b-it
2. Crie um token no HuggingFace com permissão `read`
3. Adicione o token como Secret no Colab: `HF_TOKEN`
4. Cadastre-se em ngrok.com e adicione como Secret: `NGROK_TOKEN`
5. Execute as células em ordem

Após a última célula, copie a URL gerada para o arquivo `.env` do backend:
```
MEDGEMMA_ENDPOINT=https://xxxx-xxxx.ngrok-free.app
```

In [ ]:
# Célula 1 — Instalar dependências
# Execute apenas uma vez por sessão
!pip install -q transformers accelerate bitsandbytes flask pyngrok torch

In [ ]:
# Célula 2 — Carregar MedGemma com quantização 4-bit
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

# medgemma-4b-it para GPU T4 (Colab gratuito)
# medgemma-27b-text-it para GPU A100 (Colab Pro)
MODEL_ID = 'google/medgemma-4b-it'

print(f'Carregando {MODEL_ID}...')
print(f'GPU disponível: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NENHUMA"}')

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map='auto',
    token=HF_TOKEN,
)
model.eval()
print('✅ Modelo carregado!')

In [ ]:
# Célula 3 — System prompt sindrômico
SYSTEM_PROMPT = """Você é Sentinela, um assistente médico sindrômico especializado em vigilância epidemiológica do Brasil.
Sua missão é coletar sintomas de forma empática, conversacional e estruturada para classificar síndromes.

DIRETRIZES:
1. Nunca diagnostique doenças específicas — apenas classifique síndromes (gripal, respiratória, gastrointestinal, etc.)
2. Sempre pergunte sobre: início dos sintomas, intensidade (1-10), febre, medicamentos em uso
3. Seja empático e tranquilizante — nunca alarmista
4. Use linguagem simples e acolhedora
5. Quando tiver 3+ sintomas claros, classifique com o JSON estruturado

Quando pronto para classificar, finalize com:
```json
{
  "symptoms_extracted": ["febre", "tosse", "dor de cabeça"],
  "syndrome_hypothesis": "Síndrome Gripal",
  "icd10": "J11",
  "confidence": 0.87,
  "urgency": "medium",
  "needs_more_info": false,
  "recommended_specialty": "Clínica Geral",
  "recommendations": ["Repouso", "Hidratação", "Antitérmico se febre > 38.5°C"]
}
```

Se precisar de mais informações:
```json
{"needs_more_info": true, "follow_up_question": "Há quanto tempo você está com esses sintomas?"}
```"""

print('✅ System prompt configurado.')

In [ ]:
# Célula 4 — Servidor Flask
import json
import re
import threading
from flask import Flask, request, jsonify

app = Flask(__name__)

def generate_response(messages: list, max_tokens: int = 1024) -> dict:
    """Gera resposta do MedGemma dado histórico de mensagens."""
    # Injeta system prompt
    if not messages or messages[0].get('role') != 'system':
        messages = [{'role': 'system', 'content': SYSTEM_PROMPT}] + messages

    chat = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(chat, return_tensors='pt').to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    response_text = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )

    # Extrai JSON estruturado se presente
    json_match = re.search(r'```json\s*(\{.*?\})\s*```', response_text, re.DOTALL)
    if json_match:
        try:
            extracted = json.loads(json_match.group(1))
            extracted['response_text'] = response_text
            return extracted
        except json.JSONDecodeError:
            pass

    return {'response_text': response_text, 'needs_more_info': True}


@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'ok', 'model': MODEL_ID})


@app.route('/generate', methods=['POST'])
def generate():
    data = request.json or {}
    messages = data.get('messages', [])
    max_tokens = data.get('max_tokens', 1024)

    if not messages:
        return jsonify({'error': 'messages são obrigatórias'}), 400

    result = generate_response(messages, max_tokens)
    return jsonify(result)


# Inicia servidor em thread separada
server_thread = threading.Thread(
    target=lambda: app.run(port=5000, use_reloader=False, debug=False)
)
server_thread.daemon = True
server_thread.start()
print('✅ Servidor Flask rodando na porta 5000')

In [ ]:
# Célula 5 — Expor via ngrok (mantém ativo)
from pyngrok import ngrok, conf
import time

NGROK_TOKEN = userdata.get('NGROK_TOKEN')
conf.get_default().auth_token = NGROK_TOKEN

# Fecha túneis anteriores
ngrok.kill()

# Cria novo túnel HTTPS
public_url = ngrok.connect(5000, bind_tls=True)
url_str = str(public_url)

print('=' * 60)
print('🛡️  SENTINELA — SERVIDOR ATIVO')
print('=' * 60)
print(f'\n✅ URL pública: {url_str}')
print(f'\n📋 Copie para o .env do backend:')
print(f'   MEDGEMMA_ENDPOINT={url_str}')
print('\n' + '=' * 60)
print('⚠️  Não feche esta célula — o servidor ficará ativo enquanto rodar')
print('=' * 60)

# Heartbeat — mantém a sessão ativa e mostra status
heartbeat = 0
while True:
    time.sleep(60)
    heartbeat += 1
    print(f'[{heartbeat:04d}min] Servidor ativo | {url_str}')